# Dunnhumby - Customer & Promotional Investment Analysis

## 01. Data Audit

This notebook performs the initial audit of the Dunnhumby Complete Journey dataset before building the analytical model.

### Objectives
- Inventory available datasets
- Inspect schemas and data types
- Check row counts, missing values, and duplicates
- Identify key fields and relationships
- Understand transaction coverage and time period
- Assess campaign and household coverage
- Identify data limitations relevant to the analysis

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
DATA_DIR = Path("../data")

list(DATA_DIR.iterdir())

[WindowsPath('../data/campaign_desc.csv'),
 WindowsPath('../data/campaign_table.csv'),
 WindowsPath('../data/causal_data.csv'),
 WindowsPath('../data/coupon.csv'),
 WindowsPath('../data/coupon_redempt.csv'),
 WindowsPath('../data/hh_demographic.csv'),
 WindowsPath('../data/product.csv'),
 WindowsPath('../data/transaction_data.csv')]

## 1.1 Dataset Inventory & Dimensions

Load each source table and inspect row counts, column counts, and basic structure before performing any transformations.

In [3]:
files = {
    "campaign_desc": "campaign_desc.csv",
    "campaign_table": "campaign_table.csv",
    "causal_data": "causal_data.csv",
    "coupon": "coupon.csv",
    "coupon_redeem": "coupon_redempt.csv",
    "hh_demographic": "hh_demographic.csv",
    "product": "product.csv",
    "transaction_data": "transaction_data.csv"
}

dfs = {}

for name, filename in files.items():
    dfs[name] = pd.read_csv(DATA_DIR / filename)
    
    print(
        f"{name:20} | "
        f"Rows: {dfs[name].shape[0]:,} | "
        f"Columns: {dfs[name].shape[1]}"
    )

campaign_desc        | Rows: 30 | Columns: 4
campaign_table       | Rows: 7,208 | Columns: 3
causal_data          | Rows: 36,786,524 | Columns: 5
coupon               | Rows: 124,548 | Columns: 3
coupon_redeem        | Rows: 2,318 | Columns: 4
hh_demographic       | Rows: 801 | Columns: 8
product              | Rows: 92,353 | Columns: 7
transaction_data     | Rows: 2,595,732 | Columns: 12


In [4]:
dfs["transaction_data"].head()


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0


In [5]:
dfs["transaction_data"].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2595732 entries, 0 to 2595731
Data columns (total 12 columns):
 #   Column             Dtype  
---  ------             -----  
 0   household_key      int64  
 1   BASKET_ID          int64  
 2   DAY                int64  
 3   PRODUCT_ID         int64  
 4   QUANTITY           int64  
 5   SALES_VALUE        float64
 6   STORE_ID           int64  
 7   RETAIL_DISC        float64
 8   TRANS_TIME         int64  
 9   WEEK_NO            int64  
 10  COUPON_DISC        float64
 11  COUPON_MATCH_DISC  float64
dtypes: float64(4), int64(8)
memory usage: 237.6 MB


## 1.2 Schema Inspection

Inspect the structure, data types, and sample records of each dataset to identify key fields and relationships before transformation.

In [6]:
for name, df in dfs.items():
    print(f"\n{'=' * 70}")
    print(f"{name.upper()}")
    print(f"{'=' * 70}")
    print(f"Shape: {df.shape}")
    print("\nColumns:")
    print(df.dtypes)


CAMPAIGN_DESC
Shape: (30, 4)

Columns:
DESCRIPTION    object
CAMPAIGN        int64
START_DAY       int64
END_DAY         int64
dtype: object

CAMPAIGN_TABLE
Shape: (7208, 3)

Columns:
DESCRIPTION      object
household_key     int64
CAMPAIGN          int64
dtype: object

CAUSAL_DATA
Shape: (36786524, 5)

Columns:
PRODUCT_ID     int64
STORE_ID       int64
WEEK_NO        int64
display       object
mailer        object
dtype: object

COUPON
Shape: (124548, 3)

Columns:
COUPON_UPC    int64
PRODUCT_ID    int64
CAMPAIGN      int64
dtype: object

COUPON_REDEEM
Shape: (2318, 4)

Columns:
household_key    int64
DAY              int64
COUPON_UPC       int64
CAMPAIGN         int64
dtype: object

HH_DEMOGRAPHIC
Shape: (801, 8)

Columns:
classification_1     object
classification_2     object
classification_3     object
HOMEOWNER_DESC       object
classification_5     object
classification_4     object
KID_CATEGORY_DESC    object
household_key         int64
dtype: object

PRODUCT
Shape: (92353, 7)


In [7]:
for name, df in dfs.items():
    print(f"\n{'=' * 70}")
    print(f"{name.upper()} - SAMPLE")
    print(f"{'=' * 70}")
    display(df.head(3))


CAMPAIGN_DESC - SAMPLE


,DESCRIPTION,CAMPAIGN,START_DAY,END_DAY
0,TypeB,24,659,719
1,TypeC,15,547,708
2,TypeB,25,659,691



CAMPAIGN_TABLE - SAMPLE


,DESCRIPTION,household_key,CAMPAIGN
0,TypeA,17,26
1,TypeA,27,26
2,TypeA,212,26



CAUSAL_DATA - SAMPLE


...

,classification_1,classification_2,classification_3,HOMEOWNER_DESC,classification_5,classification_4,KID_CATEGORY_DESC,household_key
0,Age Group6,X,Level4,Homeowner,Group5,2,None/Unknown,1
1,Age Group4,X,Level5,Homeowner,Group5,2,None/Unknown,7
2,Age Group2,Y,Level3,Unknown,Group4,3,1,8



PRODUCT - SAMPLE


,PRODUCT_ID,MANUFACTURER,DEPARTMENT,BRAND,COMMODITY_DESC,SUB_COMMODITY_DESC,CURR_SIZE_OF_PRODUCT
0,25671,2,GROCERY,National,FRZN ICE,ICE - CRUSHED/CUBED,22 LB
1,26081,2,MISC. TRANS.,National,NO COMMODITY DESCRIPTION,NO SUBCOMMODITY DESCRIPTION,
2,26093,69,PASTRY,Private,BREAD,BREAD:ITALIAN/FRENCH,



TRANSACTION_DATA - SAMPLE


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC
0,2375,26984851472,1,1004906,1,1.39,364,-0.6,1631,1,0.0,0.0
1,2375,26984851472,1,1033142,1,0.82,364,0.0,1631,1,0.0,0.0
2,2375,26984851472,1,1036325,1,0.99,364,-0.3,1631,1,0.0,0.0


In [8]:
for name, df in dfs.items():
    print(f"\n{name}")
    print("-" * 60)
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))


campaign_desc
------------------------------------------------------------
Shape: (30, 4)
Columns: ['DESCRIPTION', 'CAMPAIGN', 'START_DAY', 'END_DAY']

campaign_table
------------------------------------------------------------
Shape: (7208, 3)
Columns: ['DESCRIPTION', 'household_key', 'CAMPAIGN']

causal_data
------------------------------------------------------------
Shape: (36786524, 5)
Columns: ['PRODUCT_ID', 'STORE_ID', 'WEEK_NO', 'display', 'mailer']

coupon
------------------------------------------------------------
Shape: (124548, 3)
Columns: ['COUPON_UPC', 'PRODUCT_ID', 'CAMPAIGN']

coupon_redeem
------------------------------------------------------------
Shape: (2318, 4)
Columns: ['household_key', 'DAY', 'COUPON_UPC', 'CAMPAIGN']

hh_demographic
------------------------------------------------------------
Shape: (801, 8)
Columns: ['classification_1', 'classification_2', 'classification_3', 'HOMEOWNER_DESC', 'classification_5', 'classification_4', 'KID_CATEGORY_DESC', 'hou

In [9]:
for name, df in dfs.items():
    print(f"\n{name}")
    print("-" * 60)
    print(df.head(2).to_string(index=False))


campaign_desc
------------------------------------------------------------
DESCRIPTION  CAMPAIGN  START_DAY  END_DAY
      TypeB        24        659      719
      TypeC        15        547      708

campaign_table
------------------------------------------------------------
DESCRIPTION  household_key  CAMPAIGN
      TypeA             17        26
      TypeA             27        26

causal_data
------------------------------------------------------------
 PRODUCT_ID  STORE_ID  WEEK_NO display mailer
      26190       286       70       0      A
      26190       288       70       0      A

coupon
------------------------------------------------------------
 COUPON_UPC  PRODUCT_ID  CAMPAIGN
10000089061       27160         4
10000089064       27754         9

coupon_redeem
------------------------------------------------------------
 household_key  DAY  COUPON_UPC  CAMPAIGN
             1  421 10000085364         8
             1  421 51700010076         8

hh_demographic
---------

## 1.3 Data Quality Checks

The objective of this section is to identify missing values, duplicate records, and obvious data-quality issues before designing the analytical model.

In [10]:
missing_summary = []

for name, df in dfs.items():
    total_missing = df.isna().sum().sum()
    columns_with_missing = df.isna().sum()
    columns_with_missing = columns_with_missing[columns_with_missing > 0]

    missing_summary.append({
        "dataset": name,
        "total_missing_values": int(total_missing),
        "columns_with_missing": len(columns_with_missing)
    })

pd.DataFrame(missing_summary)

,dataset,total_missing_values,columns_with_missing
0,campaign_desc,0,0
1,campaign_table,0,0
2,causal_data,0,0
3,coupon,0,0
4,coupon_redeem,0,0
5,hh_demographic,0,0
6,product,0,0
7,transaction_data,0,0


### Duplicate Records

In [11]:
for name, df in dfs.items():
    print(f"{name:20} | Duplicate rows: {df.duplicated().sum():,}")

campaign_desc        | Duplicate rows: 0
campaign_table       | Duplicate rows: 0
causal_data          | Duplicate rows: 0
coupon               | Duplicate rows: 5,164
coupon_redeem        | Duplicate rows: 0
hh_demographic       | Duplicate rows: 0
product              | Duplicate rows: 0
transaction_data     | Duplicate rows: 0


In [12]:
transaction = dfs["transaction_data"]

transaction[
    ["QUANTITY", "SALES_VALUE", "RETAIL_DISC", "COUPON_DISC", "COUPON_MATCH_DISC"]
].describe().T

,count,mean,std,min,25%,50%,75%,max
QUANTITY,2595732.0,100.428558,1153.436211,0.00,1.00,1.00,1.00,89638.00
SALES_VALUE,2595732.0,3.104120,4.182274,0.00,1.29,2.00,3.49,840.00
RETAIL_DISC,2595732.0,-0.538705,1.249191,-180.00,-0.69,-0.01,0.00,3.99
COUPON_DISC,2595732.0,-0.016416,0.216841,-55.93,0.00,0.00,0.00,0.00
COUPON_MATCH_DISC,2595732.0,-0.002919,0.039690,-7.70,0.00,0.00,0.00,0.00


In [13]:
print("Negative quantity:", (transaction["QUANTITY"] < 0).sum())
print("Negative sales value:", (transaction["SALES_VALUE"] < 0).sum())

Negative quantity: 0
Negative sales value: 0


In [14]:
id_checks = {
    "households": dfs["hh_demographic"]["household_key"].nunique(),
    "transaction_households": transaction["household_key"].nunique(),
    "products": transaction["PRODUCT_ID"].nunique(),
    "stores": transaction["STORE_ID"].nunique(),
    "campaigns": dfs["campaign_desc"]["CAMPAIGN"].nunique(),
    "coupon_products": dfs["coupon"]["PRODUCT_ID"].nunique(),
    "redeemed_coupons": dfs["coupon_redeem"]["COUPON_UPC"].nunique()
}

pd.Series(id_checks)

households                  801
transaction_households     2500
products                  92339
stores                      582
campaigns                    30
coupon_products           44133
redeemed_coupons            556
dtype: int64

### Duplicate Investigation - Coupon Data

The coupon dataset contains duplicate rows. Before removing them, we inspect the duplicated records and determine whether they represent exact duplicate observations or repeated product-campaign relationships.

In [15]:
coupon = dfs["coupon"]

coupon[coupon.duplicated(keep=False)].sort_values(
    ["COUPON_UPC", "PRODUCT_ID", "CAMPAIGN"]
).head(20)

,COUPON_UPC,PRODUCT_ID,CAMPAIGN
73199,10000085426,930799,13
73748,10000085426,930799,13
62387,10000085427,983002,13
64785,10000085427,983002,13
64507,10000085427,1078612,13
64908,10000085427,1078612,13
64913,10000085427,13072690,13
64933,10000085427,13072690,13
59992,10000085427,13911346,13
64343,10000085427,13911346,13


In [16]:
coupon[
    ["COUPON_UPC", "PRODUCT_ID", "CAMPAIGN"]
].duplicated().sum()

np.int64(5164)

In [17]:
coupon.groupby(
    ["COUPON_UPC", "PRODUCT_ID", "CAMPAIGN"]
).size().sort_values(ascending=False).head(20)

COUPON_UPC   PRODUCT_ID  CAMPAIGN
57940011075  921912      27          19
57940011080  921912      27          19
57940018081  6704447     27          17
57940018076  6704447     27          17
54589314187  1024433     27          16
57940018081  7167488     27          16
57940018076  7167488     27          16
54589314175  1024433     27          16
51111122275  5565415     27          15
51111131082  1065098     27          15
57940041075  933409      27          15
51111122081  5565415     27          15
57940041055  933409      27          15
51111131075  1065098     27          15
54100027032  1118641     27          15
51111122275  1079365     27          14
             942300      27          14
54589314187  940099      27          14
51111122081  1079365     27          14
             942300      27          14
dtype: int64

### Duplicate Record Assessment

The coupon dataset contains 5,164 duplicate rows. These duplicates are also repeated across the
COUPON_UPC + PRODUCT_ID + CAMPAIGN combination, indicating exact repeated coupon-product-campaign
records rather than distinct relationships.

The records are retained at this stage because this notebook performs data auditing rather than
data transformation. Duplicate handling will be addressed during the analytical data preparation
stage after validating the expected grain of the coupon dataset.

## 1.3.4 Referential Integrity Checks

Validate whether key identifiers in child tables have corresponding records in their
parent/reference tables.

In [18]:
integrity_checks = {}

integrity_checks["transaction_households_missing"] = (
    ~transaction["household_key"].isin(
        dfs["hh_demographic"]["household_key"]
    )
).sum()

integrity_checks["transaction_products_missing"] = (
    ~transaction["PRODUCT_ID"].isin(
        dfs["product"]["PRODUCT_ID"]
    )
).sum()

integrity_checks["transaction_stores_missing"] = (
    transaction["STORE_ID"].isna()
).sum()

integrity_checks["coupon_products_missing"] = (
    ~dfs["coupon"]["PRODUCT_ID"].isin(
        dfs["product"]["PRODUCT_ID"]
    )
).sum()

integrity_checks["coupon_campaigns_missing"] = (
    ~dfs["coupon"]["CAMPAIGN"].isin(
        dfs["campaign_desc"]["CAMPAIGN"]
    )
).sum()

integrity_checks["coupon_redeem_campaigns_missing"] = (
    ~dfs["coupon_redeem"]["CAMPAIGN"].isin(
        dfs["campaign_desc"]["CAMPAIGN"]
    )
).sum()

pd.Series(integrity_checks)

transaction_households_missing     1168429
transaction_products_missing             0
transaction_stores_missing               0
coupon_products_missing                  0
coupon_campaigns_missing                 0
coupon_redeem_campaigns_missing          0
dtype: int64

## 1.3.5 Campaign Coverage

Assess campaign participation and whether campaign identifiers are consistently represented
across the campaign-related datasets.

In [19]:
campaign_desc = dfs["campaign_desc"]
campaign_table = dfs["campaign_table"]
coupon = dfs["coupon"]
coupon_redeem = dfs["coupon_redeem"]

campaign_coverage = pd.DataFrame({
    "campaign": campaign_desc["CAMPAIGN"],
    "description": campaign_desc["DESCRIPTION"],
    "households": campaign_table.groupby("CAMPAIGN")["household_key"].nunique()
        .reindex(campaign_desc["CAMPAIGN"], fill_value=0).values,
    "coupons": coupon.groupby("CAMPAIGN")["COUPON_UPC"].nunique()
        .reindex(campaign_desc["CAMPAIGN"], fill_value=0).values,
    "redemptions": coupon_redeem.groupby("CAMPAIGN")["COUPON_UPC"].count()
        .reindex(campaign_desc["CAMPAIGN"], fill_value=0).values
})

campaign_coverage

,campaign,description,households,coupons,redemptions
0,24,TypeB,100,2,10
1,15,TypeC,17,2,2
2,25,TypeB,187,17,61
3,20,TypeC,244,24,33
4,23,TypeB,183,18,60
5,21,TypeB,65,16,5
6,22,TypeB,276,21,47
7,18,TypeA,1133,209,653
8,19,TypeB,130,11,29
9,17,TypeB,202,19,45


## 1.3.6 Transaction Time Coverage

Assess the temporal coverage of transaction data and identify the observed transaction period.

In [20]:
transaction["DAY"].agg(["min", "max", "nunique"])

min          1
max        711
nunique    711
Name: DAY, dtype: int64

In [21]:
transaction.groupby("WEEK_NO")["DAY"].nunique().describe()

count    102.000000
mean       6.970588
std        0.220525
min        5.000000
25%        7.000000
50%        7.000000
75%        7.000000
max        7.000000
Name: DAY, dtype: float64

## 1.3.7 Data Coverage & Relationship Summary

Summarize the coverage of the major analytical entities and identify limitations that may
affect downstream customer, campaign, and promotional analysis.

In [22]:
coverage_summary = pd.DataFrame({
    "metric": [
        "Unique households in transactions",
        "Households in demographic data",
        "Unique products in transactions",
        "Products in product master",
        "Unique stores in transactions",
        "Campaigns",
        "Transaction days",
        "Transaction weeks",
        "Coupon records",
        "Coupon redemptions"
    ],
    "value": [
        transaction["household_key"].nunique(),
        dfs["hh_demographic"]["household_key"].nunique(),
        transaction["PRODUCT_ID"].nunique(),
        dfs["product"]["PRODUCT_ID"].nunique(),
        transaction["STORE_ID"].nunique(),
        dfs["campaign_desc"]["CAMPAIGN"].nunique(),
        transaction["DAY"].nunique(),
        transaction["WEEK_NO"].nunique(),
        len(dfs["coupon"]),
        len(dfs["coupon_redeem"])
    ]
})

coverage_summary

,metric,value
0,Unique households in transactions,2500
1,Households in demographic data,801
2,Unique products in transactions,92339
3,Products in product master,92353
4,Unique stores in transactions,582
5,Campaigns,30
6,Transaction days,711
7,Transaction weeks,102
8,Coupon records,124548
9,Coupon redemptions,2318


### Key Audit Findings

- Transaction data covers 2,500 unique households, while demographic data covers only 801 households.
- All transaction product IDs are represented in the product reference table.
- The transaction data covers 711 consecutive days across 102 weeks.
- Campaign data contains 30 campaigns.
- The coupon dataset contains duplicate records that require validation during downstream data preparation.
- Demographic information should therefore be treated as an optional enrichment rather than a complete household-level attribute source.